# 🎙️ Updated Real-Time Trimmer — Batch & Single File

> Process test audio for model evaluation.
> **Modes**: Batch folder processing, or single file analysis.

In [ ]:
%pip install -q librosa soundfile pandas numpy tqdm matplotlib

In [ ]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
import json
import shutil
import random
import re
import IPython.display as ipd

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
print('✅ All imports loaded.')

In [ ]:
# ============================================================
# CELL 3: CONFIGURATION
# ============================================================

# ==========================================================
# 🎯 CLIP DURATION — Must match your model's training duration!
# ==========================================================
TARGET_MS = 1000
# ==========================================================

# --- Mode Selection ---
# 'batch'  : Process all .wav files in a folder
# 'single' : Process one specific file
MODE = 'batch'

# --- Paths (change these) ---
# For batch mode:
BATCH_INPUT_DIR = Path(r'PUT_YOUR_TEST_AUDIO_FOLDER_PATH_HERE')
# For single mode:
SINGLE_FILE_PATH = Path(r'PUT_YOUR_SINGLE_FILE_PATH_HERE.wav')

# --- Auto-detection ---
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'Data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'Data'
OUTPUT_DIR = DATA_DIR / f'TRIMMED_REALTIME_{TARGET_MS}MS'

SAMPLE_RATE = 22050
TARGET_SAMPLES = int(SAMPLE_RATE * TARGET_MS / 1000)
HOP_MS = int(TARGET_MS * 0.5)
HOP_SAMPLES = int(SAMPLE_RATE * HOP_MS / 1000)

OVERWRITE = True

print(f'Mode            : {MODE}')
print(f'Clip Duration   : {TARGET_MS}ms = {TARGET_SAMPLES} samples')
print(f'Hop             : {HOP_MS}ms (50% overlap)')
print(f'Output          : {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELL 4: Helper Functions
# ============================================================
SAFE_NAME_RE = re.compile(r'[^A-Za-z0-9._-]+')

def sanitize_name(raw, max_len=120):
    return (SAFE_NAME_RE.sub('_', raw.strip()).strip('._') or 'clip')[:max_len]

def load_audio(path):
    y, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y.astype(np.float32)

def force_exact_length(clip):
    if len(clip) == TARGET_SAMPLES:
        return clip
    if len(clip) > TARGET_SAMPLES:
        return clip[:TARGET_SAMPLES]
    return np.pad(clip, (0, TARGET_SAMPLES - len(clip)), mode='constant')

def normalize_clip(clip):
    clip = clip - np.mean(clip)
    peak = float(np.max(np.abs(clip))) if len(clip) else 0.0
    if peak > 0.999:
        clip = clip / peak * 0.999
    return clip.astype(np.float32)

def write_clip(path, clip):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), clip, SAMPLE_RATE, subtype='PCM_16')

print('✅ Helpers defined.')

In [ ]:
# ============================================================
# CELL 5: Sliding Window Extraction
# ============================================================

def extract_all_windows(y, source_name):
    """Extract all sliding windows from an audio signal."""
    clips = []
    if len(y) < TARGET_SAMPLES:
        clip = force_exact_length(y)
        clip = normalize_clip(clip)
        clips.append((clip, 0, len(y), source_name))
        return clips
    
    for start in range(0, len(y) - TARGET_SAMPLES + 1, HOP_SAMPLES):
        end = start + TARGET_SAMPLES
        clip = y[start:end]
        clip = force_exact_length(clip)
        clip = normalize_clip(clip)
        clips.append((clip, start, end, source_name))
    return clips

print('✅ Window extraction defined.')

In [ ]:
# ============================================================
# CELL 6: RUN PIPELINE
# ============================================================

if OUTPUT_DIR.exists() and OVERWRITE:
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'reports').mkdir(exist_ok=True)

all_clips = []

if MODE == 'single':
    assert SINGLE_FILE_PATH.exists(), f'❌ File not found: {SINGLE_FILE_PATH}'
    y = load_audio(SINGLE_FILE_PATH)
    name = sanitize_name(SINGLE_FILE_PATH.stem)
    all_clips = extract_all_windows(y, name)
    print(f'\n📄 Single file: {SINGLE_FILE_PATH.name} → {len(all_clips)} clips')

elif MODE == 'batch':
    assert BATCH_INPUT_DIR.exists(), f'❌ Directory not found: {BATCH_INPUT_DIR}'
    wav_files = sorted(BATCH_INPUT_DIR.rglob('*.wav'))
    print(f'\n📁 Batch mode: {len(wav_files)} files in {BATCH_INPUT_DIR}')
    for f in tqdm(wav_files, desc='Processing files'):
        try:
            y = load_audio(f)
            name = sanitize_name(f.stem)
            clips = extract_all_windows(y, name)
            all_clips.extend(clips)
        except Exception as e:
            print(f'  ⚠️ Skipped {f.name}: {e}')

# Write all clips
manifest_rows = []
for i, (clip, start, end, src_name) in enumerate(tqdm(all_clips, desc='Writing clips')):
    clip_name = f'{src_name}_clip{i:06d}.wav'
    write_clip(OUTPUT_DIR / clip_name, clip)
    manifest_rows.append({
        'filename': clip_name, 'source': src_name,
        'start_sample': start, 'end_sample': end,
        'clip_duration_ms': TARGET_MS,
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(OUTPUT_DIR / 'reports' / 'manifest.csv', index=False)

print(f'\n✅ Done! {len(all_clips):,} clips written to {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELL 7: VERIFICATION
# ============================================================
output_files = list(OUTPUT_DIR.glob('*.wav'))
bad = []
for f in tqdm(output_files[:200], desc='Verifying'):
    y, sr = librosa.load(f, sr=None)
    if len(y) != TARGET_SAMPLES:
        bad.append((f.name, len(y)))

if bad:
    print(f'⚠️ {len(bad)} files wrong length!')
else:
    print(f'✅ Checked {min(200, len(output_files))} files — all exactly {TARGET_SAMPLES} samples ({TARGET_MS}ms).')
print(f'📊 Total clips: {len(output_files):,}')